# NB14 — Cross-Validation Harness (Priority 2)

Builds a reusable 5-fold CV harness on the merged Oct+Dec training set
(`merged_oct_dec_clean_v1.parquet`, 1,003,129 rows). Per the roadmap:

- **Feature engineer refit inside each fold** — target encodings are fitted
  statistics; fitting once on full data before CV leaks. FE is fit on each
  fold's training portion only.
- **Cross-validate all quantiles, not just p50** — per fold we compute pinball
  loss for q50 and q85 *and* empirical Q15–Q85 coverage.
- **Report mean ± std per metric** — changes are only accepted if the
  improvement exceeds fold noise.
- **Training budget: early stopping** (lr=0.1, n_estimators capped high, early
  stopping on a fold-internal validation slice).

This establishes the V2 baseline under CV. Final client-facing numbers always
come from the future out-of-time extraction, never from CV.


In [1]:
import os
import sys
import json
import importlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import mlflow
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import features.feature_engineering as _fe_module
importlib.reload(_fe_module)
from features.feature_engineering import CarPriceFeatureEngineer

# ── Constants ────────────────────────────────────────────────────────────────
CURRENT_YEAR   = 2025
SEED           = 42
N_FOLDS        = 5
ES_VAL_FRAC    = 0.10          # fold-internal slice for early stopping
ES_ROUNDS      = 50            # early-stopping patience
N_ESTIMATORS   = 5000          # cap; early stopping decides the real count
MLFLOW_URI     = f"file://{project_root / 'mlruns'}"
MLFLOW_EXP     = 'nb14_cv_harness'

DATA_PATH = project_root / 'data' / 'processed' / 'merged_oct_dec_clean_v1.parquet'

# ── Lean feature set (identical 31 cols to NB06/NB12 production model) ─────────
LEAN_FEATURES = [
    'car_age',
    'brand_avg_age',       'brand_median_age',
    'brand_mean_log_price','brand_median_log_price','brand_std_log_price',
    'model_count',         'model_popularity_ratio',
    'model_mean_log_price','model_median_log_price','model_std_log_price',
    'is_almost_new',       'decade',
    'sqrt_age',            'age_squared',           'age_cubed',
    'brand_top25_price',   'brand_bottom25_price',  'brand_top5_price',
    'model_top25_price',   'model_bottom25_price',  'model_top5_price',
    'model_rank_within_brand',
    'brand_p25_log_price', 'brand_p75_log_price',   'brand_p90_log_price', 'brand_iqr_log_price',
    'model_p25_log_price', 'model_p75_log_price',   'model_p90_log_price', 'model_iqr_log_price',
]

# ── LightGBM hyperparameters (lr=0.1 as NB06/NB12; early stopping replaces fixed n) ─
LGB_PARAMS = dict(
    objective     = 'quantile',
    learning_rate = 0.1,
    n_estimators  = N_ESTIMATORS,
    random_state  = SEED,
    verbose       = -1,
)
QUANTILES = [(0.15, 'q15'), (0.50, 'q50'), (0.85, 'q85')]

print(f'Project root : {project_root}')
print(f'Data         : {DATA_PATH.name}')
print(f'Folds        : {N_FOLDS}  |  early-stop rounds: {ES_ROUNDS}  |  n_est cap: {N_ESTIMATORS}')
print(f'Lean features: {len(LEAN_FEATURES)}')
print(f'MLflow       : {MLFLOW_URI} -> {MLFLOW_EXP}')
print('✓ Setup complete')


Project root : /Users/brunobrumbrum/car_price_prediction
Data         : merged_oct_dec_clean_v1.parquet
Folds        : 5  |  early-stop rounds: 50  |  n_est cap: 5000
Lean features: 31
MLflow       : file:///Users/brunobrumbrum/car_price_prediction/mlruns -> nb14_cv_harness
✓ Setup complete


In [2]:
# ── Load merged training set ──────────────────────────────────────────────────
df = pl.read_parquet(DATA_PATH)
print(f'Loaded {df.height:,} rows × {df.width} cols')
print(f'Columns : {df.columns}')
print(f'Year range: {df["year"].min()}–{df["year"].max()}')
print(f'Price range: €{df["price"].min():,.0f}–€{df["price"].max():,.0f}')
assert {'price','year','km','brand','model','log_price'} <= set(df.columns)
print('✓ Schema OK')


Loaded 1,003,129 rows × 8 cols
Columns : ['price', 'year', 'km', 'brand', 'model', 'energie', 'horsepower', 'log_price']
Year range: 1990–2025
Price range: €500–€250,000
✓ Schema OK


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def select_lean(df_fe, feat_cols=LEAN_FEATURES):
    """Polars FE output -> pandas matrix with exactly feat_cols, missing filled 0."""
    X = df_fe.to_pandas().fillna(0)
    for c in feat_cols:
        if c not in X.columns:
            X[c] = 0
    return X[feat_cols].copy()

def pinball_loss(y_true, y_pred, alpha):
    """Pinball loss in log space (y_true, y_pred are log prices)."""
    e = y_true - y_pred
    return float(np.mean(np.where(e >= 0, alpha * e, (alpha - 1) * e)))

AGE_BUCKETS = [
    (0,  2,  '0-2 yr (2023-25)'),
    (3,  5,  '3-5 yr (2020-22)'),
    (6,  10, '6-10 yr (2015-19)'),
    (11, 15, '11-15 yr (2010-14)'),
    (16, 20, '16-20 yr (2005-09)'),
    (21, 99, '21+ yr (pre-2005)'),
]
def age_to_bucket(age):
    for lo, hi, lab in AGE_BUCKETS:
        if lo <= age <= hi:
            return lab
    return 'unknown'

print('✓ Helpers defined')


✓ Helpers defined


In [4]:
# ── Per-fold training routine ─────────────────────────────────────────────────
# CRITICAL: the FeatureEngineer is fit ONLY on each fold's model-training rows.
# Fitting on full data (or even on the fold-val rows) leaks target encodings.
def run_fold(df_train_pl, df_val_pl, fold_id):
    """Fit FE on fold-train, train q15/q50/q85 with early stopping, predict on fold-val.

    Returns dict with per-quantile log predictions on the val fold and the
    chosen best_iteration per quantile.
    """
    # Carve a fold-internal early-stopping slice from the training rows only.
    tr_pd = df_train_pl.to_pandas()
    es_tr_idx, es_val_idx = train_test_split(
        np.arange(len(tr_pd)), test_size=ES_VAL_FRAC, random_state=SEED,
    )
    df_es_tr  = pl.from_pandas(tr_pd.iloc[es_tr_idx].reset_index(drop=True))
    df_es_val = pl.from_pandas(tr_pd.iloc[es_val_idx].reset_index(drop=True))

    # Fit FE on the model-training rows ONLY (no es-val, no fold-val).
    fe = CarPriceFeatureEngineer(
        current_year=CURRENT_YEAR, brand_onehot=False, model_onehot=False,
    )
    fe.fit(df_es_tr.drop(['price', 'log_price']), df_es_tr['price'])

    X_tr  = select_lean(fe.transform(df_es_tr.drop(['price', 'log_price'])))
    X_es  = select_lean(fe.transform(df_es_val.drop(['price', 'log_price'])))
    X_val = select_lean(fe.transform(df_val_pl.drop(['price', 'log_price'])))

    y_tr  = df_es_tr['log_price'].to_numpy()
    y_es  = df_es_val['log_price'].to_numpy()

    preds_log = {}
    best_iters = {}
    for alpha, label in QUANTILES:
        m = LGBMRegressor(alpha=alpha, **LGB_PARAMS)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_es, y_es)],
            eval_metric='quantile',
            callbacks=[early_stopping(ES_ROUNDS, verbose=False)],
        )
        preds_log[label] = m.predict(X_val)
        best_iters[label] = int(m.best_iteration_ or N_ESTIMATORS)
    return preds_log, best_iters

print('✓ Fold routine defined')


✓ Fold routine defined


In [5]:
# ── Run 5-fold CV ─────────────────────────────────────────────────────────────
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
idx_all = np.arange(df.height)

fold_rows   = []   # aggregate metrics per fold
seg_records = []   # per-fold per-segment rows (year + bucket), for stability checks

t0 = datetime.now()
for fold_id, (tr_idx, val_idx) in enumerate(kf.split(idx_all)):
    df_tr  = df[tr_idx.tolist()]
    df_val = df[val_idx.tolist()]

    preds_log, best_iters = run_fold(df_tr, df_val, fold_id)

    y_val_log = df_val['log_price'].to_numpy()
    y_val_eur = df_val['price'].to_numpy()
    years_val = df_val['year'].to_numpy()
    p_eur = {lab: np.exp(preds_log[lab]) for _, lab in QUANTILES}

    pl_q50 = pinball_loss(y_val_log, preds_log['q50'], 0.50)
    pl_q85 = pinball_loss(y_val_log, preds_log['q85'], 0.85)
    pl_q15 = pinball_loss(y_val_log, preds_log['q15'], 0.15)
    cov    = float(np.mean((p_eur['q15'] <= y_val_eur) & (y_val_eur <= p_eur['q85'])) * 100)
    medape = float(np.median(np.abs(y_val_eur - p_eur['q50']) / np.clip(y_val_eur, 1, None) * 100))
    mae    = float(mean_absolute_error(y_val_eur, p_eur['q50']))

    fold_rows.append({
        'fold': fold_id, 'n_val': len(val_idx),
        'pinball_q15': pl_q15, 'pinball_q50': pl_q50, 'pinball_q85': pl_q85,
        'coverage_q15_q85': cov, 'median_ape': medape, 'mae_eur': mae,
        'best_iter_q15': best_iters['q15'], 'best_iter_q50': best_iters['q50'],
        'best_iter_q85': best_iters['q85'],
    })

    # Per-segment (age bucket) breakdown for this fold
    ages = CURRENT_YEAR - years_val
    buckets = np.array([age_to_bucket(a) for a in ages])
    for _, _, lab in AGE_BUCKETS:
        mask = buckets == lab
        if mask.sum() < 50:
            continue
        seg_records.append({
            'fold': fold_id, 'bucket': lab, 'n': int(mask.sum()),
            'median_ape': float(np.median(np.abs(y_val_eur[mask] - p_eur['q50'][mask])
                                          / np.clip(y_val_eur[mask], 1, None) * 100)),
            'coverage': float(np.mean((p_eur['q15'][mask] <= y_val_eur[mask])
                                      & (y_val_eur[mask] <= p_eur['q85'][mask])) * 100),
        })

    el = (datetime.now() - t0).total_seconds()
    print(f'Fold {fold_id}: n_val={len(val_idx):,}  '
          f'pinball_q50={pl_q50:.5f}  pinball_q85={pl_q85:.5f}  '
          f'cov={cov:.1f}%  medAPE={medape:.2f}%  '
          f'iters(q50)={best_iters["q50"]}  [{el:.0f}s]')

cv_df = pd.DataFrame(fold_rows)
seg_df = pd.DataFrame(seg_records)
print(f'\n✓ CV complete in {(datetime.now()-t0).total_seconds():.0f}s')


Fold 0: n_val=200,626  pinball_q50=0.09913  pinball_q85=0.05641  cov=69.6%  medAPE=13.44%  iters(q50)=1724  [87s]


Fold 1: n_val=200,626  pinball_q50=0.09855  pinball_q85=0.05621  cov=69.7%  medAPE=13.35%  iters(q50)=1988  [187s]


Fold 2: n_val=200,626  pinball_q50=0.09907  pinball_q85=0.05638  cov=69.4%  medAPE=13.41%  iters(q50)=1250  [259s]


Fold 3: n_val=200,626  pinball_q50=0.09819  pinball_q85=0.05605  cov=69.7%  medAPE=13.29%  iters(q50)=1673  [345s]


Fold 4: n_val=200,625  pinball_q50=0.09818  pinball_q85=0.05615  cov=69.8%  medAPE=13.37%  iters(q50)=1732  [420s]

✓ CV complete in 420s


In [6]:
# ── Aggregate: mean ± std per metric ──────────────────────────────────────────
metric_cols = ['pinball_q15', 'pinball_q50', 'pinball_q85',
               'coverage_q15_q85', 'median_ape', 'mae_eur',
               'best_iter_q50']
summary = cv_df[metric_cols].agg(['mean', 'std']).T
summary.columns = ['mean', 'std']

print('='*64)
print(f'NB14 — {N_FOLDS}-FOLD CV SUMMARY (merged Oct+Dec, lean spec)')
print('='*64)
for m in metric_cols:
    mu, sd = summary.loc[m, 'mean'], summary.loc[m, 'std']
    if m == 'mae_eur':
        print(f'  {m:20s}: €{mu:,.0f}  ± €{sd:,.0f}')
    elif m == 'best_iter_q50':
        print(f'  {m:20s}: {mu:,.0f}  ± {sd:,.0f}  (early-stopped trees)')
    elif 'coverage' in m or 'ape' in m:
        print(f'  {m:20s}: {mu:.2f}%  ± {sd:.2f}%')
    else:
        print(f'  {m:20s}: {mu:.5f}  ± {sd:.5f}')

print('\nPer-fold detail:')
display(cv_df.round({'pinball_q15':5,'pinball_q50':5,'pinball_q85':5,
                     'coverage_q15_q85':2,'median_ape':2,'mae_eur':0}))


NB14 — 5-FOLD CV SUMMARY (merged Oct+Dec, lean spec)
  pinball_q15         : 0.06019  ± 0.00023
  pinball_q50         : 0.09863  ± 0.00046
  pinball_q85         : 0.05624  ± 0.00015
  coverage_q15_q85    : 69.66%  ± 0.17%
  median_ape          : 13.37%  ± 0.06%
  mae_eur             : €2,325  ± €13
  best_iter_q50       : 1,673  ± 267  (early-stopped trees)

Per-fold detail:


,fold,n_val,pinball_q15,pinball_q50,pinball_q85,coverage_q15_q85,median_ape,mae_eur,best_iter_q15,best_iter_q50,best_iter_q85
0,0,200626,0.06043,0.09913,0.05641,69.58,13.44,2340.0,1537,1724,1161
1,1,200626,0.06030,0.09855,0.05621,69.72,13.35,2318.0,1283,1988,1858
2,2,200626,0.06034,0.09907,0.05638,69.40,13.41,2339.0,864,1250,1720
3,3,200626,0.05998,0.09819,0.05605,69.74,13.29,2316.0,1367,1673,1407
4,4,200625,0.05990,0.09818,0.05615,69.84,13.37,2312.0,923,1732,1449


In [7]:
# ── Per-segment stability across folds (age bucket) ───────────────────────────
# A metric is only trustworthy if it is stable across folds. Report mean ± std
# of median APE and coverage per age bucket.
bucket_order = [lab for _, _, lab in AGE_BUCKETS]
seg_summary = (seg_df.groupby('bucket')
               .agg(n_mean=('n', 'mean'),
                    median_ape_mean=('median_ape', 'mean'),
                    median_ape_std=('median_ape', 'std'),
                    coverage_mean=('coverage', 'mean'),
                    coverage_std=('coverage', 'std'))
               .reindex(bucket_order))

print('Per-age-bucket CV stability (mean ± std across folds):')
print('='*78)
for lab in bucket_order:
    if lab not in seg_summary.index or pd.isna(seg_summary.loc[lab, 'n_mean']):
        continue
    r = seg_summary.loc[lab]
    print(f'  {lab:20s}  n≈{r["n_mean"]:>7,.0f}  '
          f'medAPE {r["median_ape_mean"]:5.2f}% ± {r["median_ape_std"]:4.2f}  '
          f'cov {r["coverage_mean"]:5.1f}% ± {r["coverage_std"]:4.1f}')
display(seg_summary.round(2))


Per-age-bucket CV stability (mean ± std across folds):
  0-2 yr (2023-25)      n≈ 30,959  medAPE  8.16% ± 0.07  cov  69.7% ±  0.3
  3-5 yr (2020-22)      n≈ 41,066  medAPE  9.40% ± 0.08  cov  69.7% ±  0.3
  6-10 yr (2015-19)     n≈ 54,255  medAPE 12.89% ± 0.06  cov  69.7% ±  0.3
  11-15 yr (2010-14)    n≈ 37,002  medAPE 18.28% ± 0.15  cov  69.7% ±  0.5
  16-20 yr (2005-09)    n≈ 24,020  medAPE 25.87% ± 0.25  cov  69.6% ±  0.2
  21+ yr (pre-2005)     n≈ 13,324  medAPE 34.76% ± 0.31  cov  69.0% ±  0.3


,n_mean,median_ape_mean,median_ape_std,coverage_mean,coverage_std
bucket,,,,,
0-2 yr (2023-25),30959.2,8.16,0.07,69.68,0.29
3-5 yr (2020-22),41066.0,9.40,0.08,69.74,0.31
6-10 yr (2015-19),54254.8,12.89,0.06,69.71,0.25
11-15 yr (2010-14),37001.8,18.28,0.15,69.73,0.53
16-20 yr (2005-09),24019.8,25.87,0.25,69.62,0.22
21+ yr (pre-2005),13324.2,34.76,0.31,68.96,0.26


In [8]:
# ── Log to MLflow ─────────────────────────────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXP)

with mlflow.start_run(run_name='cv5_lean_baseline'):
    mlflow.log_params({
        'n_folds':       N_FOLDS,
        'n_rows':        df.height,
        'n_features':    len(LEAN_FEATURES),
        'feature_spec':  'lean_31',
        'lgb_lr':        LGB_PARAMS['learning_rate'],
        'lgb_n_est_cap': N_ESTIMATORS,
        'early_stop_rounds': ES_ROUNDS,
        'es_val_frac':   ES_VAL_FRAC,
        'fe_refit':      'per_fold',
        'dataset':       'merged_oct_dec_clean_v1',
        'cv_budget':     'early_stopping',
    })
    for m in metric_cols:
        mlflow.log_metric(f'{m}_mean', round(float(summary.loc[m, 'mean']), 5))
        mlflow.log_metric(f'{m}_std',  round(float(summary.loc[m, 'std']), 5))
    # per-fold raw values too
    for _, row in cv_df.iterrows():
        f = int(row['fold'])
        for m in ['pinball_q50', 'pinball_q85', 'coverage_q15_q85', 'median_ape']:
            mlflow.log_metric(m, round(float(row[m]), 5), step=f)
    print('✓ Logged cv5_lean_baseline to MLflow')

print(f'\nExperiment: {MLFLOW_EXP}')
print(f'View: mlflow ui --backend-store-uri {MLFLOW_URI} --port 5001')


/Users/brunobrumbrum/car_price_prediction/venv_cars/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/06/09 23:11:33 INFO mlflow.tracking.fluent: Experiment with name 'nb14_cv_harness' does not exist. Creating a new experiment.


✓ Logged cv5_lean_baseline to MLflow

Experiment: nb14_cv_harness
View: mlflow ui --backend-store-uri file:///Users/brunobrumbrum/car_price_prediction/mlruns --port 5001


In [9]:
# ── Persist CV results to disk (CSV, for the next notebooks to compare against) ─
cv_out = project_root / 'output'
cv_out.mkdir(exist_ok=True)
cv_df.to_csv(cv_out / 'nb14_cv_folds.csv', index=False)
seg_summary.to_csv(cv_out / 'nb14_cv_segment_stability.csv')
summary.to_csv(cv_out / 'nb14_cv_summary.csv')

baseline = {
    'created':     datetime.now().isoformat(),
    'dataset':     'merged_oct_dec_clean_v1.parquet',
    'n_rows':      int(df.height),
    'n_folds':     N_FOLDS,
    'feature_spec':'lean_31',
    'cv_budget':   'early_stopping (lr=0.1, n_est cap=%d, patience=%d)' % (N_ESTIMATORS, ES_ROUNDS),
    'metrics_mean_std': {m: [round(float(summary.loc[m,'mean']),5),
                             round(float(summary.loc[m,'std']),5)] for m in metric_cols},
}
with open(cv_out / 'nb14_cv_baseline.json', 'w') as f:
    json.dump(baseline, f, indent=2)

print('Saved:')
for p in ['nb14_cv_folds.csv','nb14_cv_segment_stability.csv',
          'nb14_cv_summary.csv','nb14_cv_baseline.json']:
    print(f'  output/{p}')
print('\n✓ NB14 complete — this CV baseline is the reference for NB15+ comparisons.')


Saved:
  output/nb14_cv_folds.csv
  output/nb14_cv_segment_stability.csv
  output/nb14_cv_summary.csv
  output/nb14_cv_baseline.json

✓ NB14 complete — this CV baseline is the reference for NB15+ comparisons.
